In [23]:
# conda activate genomic_tools

import sys
import pysam
import pickle
import pandas as pd
from pyfaidx import Fasta
from collections import defaultdict

sys.path.append("code")

from se_event_utils import *

pd.set_option('display.max_colwidth', None)

In [ ]:
ct_SEs = pickle.load(open(f"data/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_ct_SEs.pkl", "rb"))

event_dicts = pickle.load(open('data/event_dicts.pkl', 'rb'))
event_info = pickle.load(open('data/event_info.pkl', 'rb'))

cds_by_transcript= pickle.load(open("data/gencode.v50.annotation_cds_by_transcript.pkl", "rb"))
exons_by_transcript = pickle.load(open("data/gencode.v50.annotation_exons_by_transcript.pkl", "rb"))

proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v50.pc_translations.fa")

In [25]:
# Get all cell type splicing events
events_by_ct = {}

for ct, rec in ct_SEs.items():
    evs = rec.index.tolist()
    events_by_ct[ct] = evs 
    
all_ct_events = set().union(*events_by_ct.values())

In [26]:
with open("data/events_by_ct.pkl", "wb") as file:
    pickle.dump(events_by_ct, file)

In [27]:
len(all_ct_events)

12149

In [ ]:
event_by_coords = {(ed['es'], ed['ee']): ed['event'] for ed in event_dicts} # for boundary siblings: one event per unique (es, ee)

event_by_coords_list = defaultdict(list) # for junction siblings: multiple events can share same (es, ee)
for ed in event_dicts:
    event_by_coords_list[(ed['es'], ed['ee'])].append(ed['event'])

tags_to_exclude = {'mRNA_start_NF', 'cds_start_NF', 'mRNA_end_NF', 'cds_end_NF'}

def has_incomplete_tag(tag):
    return any(t in str(tag) for t in tags_to_exclude)

MIN_AA = 30

event_protein_map  = {}

for ev, rec in event_info.items():
    if ev not in all_ct_events:
        continue

    ####################################################################    
    # add sequence of top-scoring transcript
    comp = {t: d for t, d in rec['compatible'].items()
            if isinstance(d.get('overlap_type'), str)
            and d['overlap_type'] != 'noncoding_or_utr'
            and d.get('transcript_type') == 'protein_coding'
            and not has_incomplete_tag(d.get('transcript_tag', ''))
        }
    if not comp:
        continue
    
    best_t = max(comp, key=lambda t: score_transcript(t, comp[t]))
    d = comp[best_t]
    chr = rec['meta']['chr']

    event_protein_map[ev] = {
        'meta': rec['meta'],
        'inclusion': {
            'transcript_id': best_t,
            'aa_start': d['aa_start'],
            'aa_end': d['aa_end'],
            'exon_cds_start': d['exon_cds_start'],
            'exon_cds_end': d['exon_cds_end'],
            'frame_preserving': d['frame_preserving'],
            'clean_start': d['clean_start'],
            'clean_end': d['clean_end']
        },
        'real_skip': None,
        'exon_diff_boundary_siblings': None,
        'exon_diff_junction_siblings': None
    }
    
    ####################################################################    
    # add sequence of top-scoring transcript where the event was SKIPPED
    skip_candidates = {}
    excluded_ttypes = set()
    excluded_tags = set()
    
    for t, skip_d in rec['exon_skipped'].items():
        if not isinstance(skip_d, dict):
            continue
        exons = exons_by_transcript.get(t)
        if exons is None:
            continue
        ttype = exons.iloc[0].get('transcript_type', '')
        tag = exons.iloc[0].get('tag', '')
        if ttype != 'protein_coding' or has_incomplete_tag(tag):
            excluded_ttypes.add(ttype)
            excluded_tags.add(tag)
            continue
        skip_candidates[t] = {
            'transcript_type': ttype,
            'transcript_tag': tag,
            'coding_nt_length': get_coding_nt_length(t, cds_by_transcript),
            'is_called_sibling': skip_d.get('is_called_sibling', False)
        }
    
    if skip_candidates:
        # prefer siblings (transcripts compatible with emitted events), but can fall back to any protein-coding transcript
        called = {t: v for t, v in skip_candidates.items() if v['is_called_sibling']}
        pool = called if called else skip_candidates
        best_skip_t = max(pool, key=lambda t: score_transcript(t, pool[t]))
        event_protein_map[ev]['real_skip'] = {
            'transcript_id': best_skip_t,
            'excluded_transcript_types': excluded_ttypes,
            'excluded_transcript_tags': excluded_tags,
        }
    else:
        event_protein_map[ev]['real_skip'] = {
            'transcript_id': None,
            'excluded_transcript_types': excluded_ttypes,
            'excluded_transcript_tags': excluded_tags,
        }

    ####################################################################    
    # add sequence for (top-scoring) sibling transcript with BOUNDARY VARIANTS
    
    best_entry = None
    best_score = None
    seen_sib_evs = set()

    for sib_t, sib in rec['exon_diff_boundary'].items():
        if not sib.get('is_called_sibling'):
            continue
        
        # all events compatible with exon boundaires
        sib_evs = [e for e in event_by_coords_list.get((sib['start'], sib['end']), [])
                   if e in event_info]
        
        for sib_ev in sib_evs:
            if sib_ev in seen_sib_evs:
                continue
            seen_sib_evs.add(sib_ev)
        
            # note to self: previously logged transcripts for sibling events, but those transcripts 
            # were only checked for matching exon boundaries, NOT flanking introns. 
            # let's to do that now by enforcing that sibling transcripts were identified as 'compatible' in another event
            all_compatible = event_info[sib_ev]['compatible']
            sib_comp = {t: d for t, d in all_compatible.items()
                        if isinstance(d.get('overlap_type'), str)
                        and d['overlap_type'] != 'noncoding_or_utr'
                        and d.get('transcript_type') == 'protein_coding'
                        and not has_incomplete_tag(d.get('transcript_tag', ''))
                    }                        
            
            excluded_ttypes = set()
            excluded_tags = set()
            for t, d in all_compatible.items():
                if t not in sib_comp:
                    excluded_ttypes.add(d.get('transcript_type', ''))
                    excluded_tags.add(d.get('transcript_tag', ''))

            if sib_comp:
                best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
                sib_d = sib_comp[best_sib_t]
                score = score_transcript(best_sib_t, sib_d)
                if best_score is None or score > best_score:
                    best_score = score
                    best_entry = {
                        'transcript_id': best_sib_t,
                        'aa_start': sib_d['aa_start'],
                        'aa_end': sib_d['aa_end'],
                        'exon_cds_start': sib_d['exon_cds_start'],
                        'exon_cds_end': sib_d['exon_cds_end'],
                        'frame_preserving': sib_d['frame_preserving'],
                        'clean_start': sib_d['clean_start'],
                        'clean_end': sib_d['clean_end'],
                        'excluded_transcript_types': excluded_ttypes,
                        'excluded_transcript_tags': excluded_tags,
                    }

            elif best_entry is None:
                best_entry = {
                    'transcript_id': None,
                    'excluded_transcript_types': excluded_ttypes,
                    'excluded_transcript_tags': excluded_tags,
                }
            
    event_protein_map[ev]['exon_diff_boundary_siblings'] = best_entry
     
    ####################################################################    
    # add sequence for (top-scoring) sibling transcript with JXN VARIANT
    best_entry = None
    best_score = None
    seen_sib_evs = set()
        
    for sib_t, sib in rec['exon_diff_junction'].items():
        if not sib.get('is_called_sibling'):
            continue
        
        # need to remove parent inclusion transcript since exon boundaries will be identical
        sib_evs = [e for e in event_by_coords_list.get((rec['meta']['es'], rec['meta']['ee']), [])
                   if e != ev and e in event_info]
        
        for sib_ev in sib_evs:
            if sib_ev in seen_sib_evs:
                continue
            seen_sib_evs.add(sib_ev)
            
            # note to self: previously logged transcripts for sibling events, but those transcripts 
            # were only checked for matching exon boundaries, NOT flanking introns. 
            # let's to do that now by enforcing that sibling transcripts were identified as 'compatible' in another event
            all_compatible = event_info[sib_ev]['compatible']
            sib_comp = {t: d for t, d in all_compatible.items()
                        if isinstance(d.get('overlap_type'), str)
                        and d['overlap_type'] != 'noncoding_or_utr'
                        and d.get('transcript_type') == 'protein_coding'
                        and not has_incomplete_tag(d.get('transcript_tag', ''))
                    } 
            
            excluded_ttypes = set()
            excluded_tags = set()
            for t, d in all_compatible.items():
                if t not in sib_comp:
                    excluded_ttypes.add(d.get('transcript_type', ''))
                    excluded_tags.add(d.get('transcript_tag', ''))
                    
            if sib_comp:
                best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
                sib_d = sib_comp[best_sib_t]
                score = score_transcript(best_sib_t, sib_d)
                if best_score is None or score > best_score:
                    best_score = score       
                    best_entry = {
                        'transcript_id': best_sib_t,
                        'aa_start': sib_d['aa_start'],
                        'aa_end': sib_d['aa_end'],
                        'exon_cds_start': sib_d['exon_cds_start'],
                        'exon_cds_end': sib_d['exon_cds_end'],
                        'frame_preserving': sib_d['frame_preserving'],
                        'clean_start': sib_d['clean_start'],
                        'clean_end': sib_d['clean_end'],
                        'excluded_transcript_types': excluded_ttypes,
                        'excluded_transcript_tags': excluded_tags
                    }

            elif best_entry is None:
                best_entry = {
                    'transcript_id': None,
                    'excluded_transcript_types': excluded_ttypes,
                    'excluded_transcript_tags': excluded_tags,
                }
                
    event_protein_map[ev]['exon_diff_junction_siblings'] = best_entry

In [39]:
event_info[sib_ev]['compatible']

{'ENST00000494752': {'overlap_type': 'noncoding_or_utr',
  'transcript_type': 'protein_coding_CDS_not_defined',
  'transcript_tag': ''}}

In [41]:
sib_d.get("aa_start")

In [22]:
d

{'aa_start': 124,
 'aa_end': 132,
 'coding_nt_length': 26,
 'overlap_type': 'fully_coding',
 'gtf_frame': 0,
 'clean_start': True,
 'clean_end': False,
 'frame_preserving': False,
 'transcript_type': 'protein_coding',
 'transcript_tag': 'basic,Ensembl_canonical,GENCODE_Primary,MANE_Select,appris_principal_1,CCDS'}

In [11]:
transcripts = set()
for info in event_protein_map.values():
    if info.get('inclusion'):
        t = info['inclusion'].get('transcript_id')
        if t:
            transcripts.add(t)
            
        if info.get('real_skip'):
            t = info['real_skip'].get('transcript_id')
            if t:
                transcripts.add(t)
        sib = info.get('exon_diff_boundary_siblings')
        if sib:
            t = sib.get('transcript_id')
            if t:
                transcripts.add(t)

        sib = info.get('exon_diff_junction_siblings')
        if sib:
            t = sib.get('transcript_id')
            if t:
                transcripts.add(t)

In [12]:
len(transcripts)

14917

In [13]:
len(set(transcripts))

14917

In [53]:
# Build a dict transcript <--> protein dict

protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst = parts[1].split(".")[0]
    protein_by_transcript[enst] = str(proteins[key])

In [ ]:
# write protein sequences

modified_transcript_products = {}
transcripts_with_seqs = []

with open("data/proteins_v2.fa", "w") as f:
    for t in protein_by_transcript.keys(): 
        if t in transcripts:
            transcripts_with_seqs.append(t)
            seq = str(protein_by_transcript[t]).rstrip("*")
            if "X" in seq:
                # some transcripts have AA placeholders, track where the placeholder was
                modified_transcript_products[t] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{t}\n{seq}\n")

In [ ]:
len(modified_transcript_products)

In [ ]:
len(set(transcripts_with_seqs))

14917

In [51]:
with open("data/event_protein_map.pkl", "wb") as file:
    pickle.dump(event_protein_map, file)
    
# with open("data/modified_transcript_products.pkl", "wb") as file:
#     pickle.dump(modified_transcript_products, file)